# Configurando el ambiente

In [1]:
import sys

In [2]:
IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    %pip install -q torchmetrics

In [13]:
import torch
import torch
import torchvision 

In [14]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

# Importando los datos

In [ ]:
import torchvision.transforms.v2 as T

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_valid_data = torchvision.datasets.CIFAR10(root="datasets", train=True, download=True, transform=toTensor)
test_data = torchvision.datasets.CIFAR10(root="datasets", train=False, download=True, transform=toTensor)

torch.manual_seed(21)
train_size = len(train_valid_data) * 90 // 100
valid_size = len(train_valid_data) - train_size
train_data, valid_data = torch.utils.data.random_split(train_valid_data, lengths=[train_size, valid_size])

/home/isa/Documents/ML/CIFAR10/.venv/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)

# Definiendo el modelo

In [5]:
import torch.nn as nn
import torchmetrics

In [6]:
hidden_layer_size = 100
hidden_layers = 20
activation_functions = nn.ReLU()

In [7]:
def use_he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        nn.init.zeros_(module.bias) # no importa mucho si inicializa en 0s 

In [8]:
def build_image_model(n_in, n_hidden_layers, n_hidden_neurons, n_out):

        layers = [nn.Flatten(), nn.Linear(n_in, n_hidden_neurons), nn.SiLU()] # swish suele mejor para modelos profundos
        for _ in range(n_hidden_layers - 1):
            layers += [nn.Linear(n_hidden_neurons, n_hidden_neurons), nn.SiLU()]
        
        layers += [nn.Linear(n_hidden_neurons, n_out)]

        model = torch.nn.Sequential(*layers)
        model.apply(use_he_init)
        return model

In [9]:
from functools import reduce
X_sample, _ = test_data[0]

in_dims = reduce(lambda x, y: x*y, X_sample[0].shape)
out_dims = len(test_data.classes)

In [10]:
way_too_big = build_image_model(in_dims, 20, 100, out_dims)

In [ ]:
def train_with_early_stopping(model, optimizer, criterion, metric, train_loader, valid_loader, 
                              max_epochs, patience_epochs=5, checkpoint_path=None, scheduler=None):
    metric.reset()
    checkpoint_path = checkpoint_path or "my_checkpoint.pt"
    best_valid_loss = float("inf")
    loss_histories = {"train_losses": [], "validation_losses": []}
    metric_histories = {"train_metrics": [], "validation_metrics": []}
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        metric.reset()
        total_train_loss = 0
        # Entrenamiento
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            batch_loss = criterion(preds, y_batch)
            total_train_loss += batch_loss.item()
            batch_loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(preds, y_batch)
        train_metric = metric.compute().item()

        # Prueba en conjunto de validación
        total_validation_loss = 0
        model.eval() # importante cambiar a modo de evaluación
        metric.reset() # Reseteamos las métricas entre conjuntos de entrenamiento y validación
        with torch.no_grad():
            for X_batch, y_batch in valid_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds = model(X_batch)
                batch_loss = criterion(preds, y_batch)
                total_validation_loss += batch_loss.item()
                metric.update(preds, y_batch)
        validation_metric = metric.compute().item()

        # Calculamos la pérdida para ambos conjuntos 
        mean_train_loss = total_train_loss / len(train_loader)
        mean_validation_loss = total_validation_loss / len(valid_loader)

        loss_histories["train_losses"].append(mean_train_loss)
        loss_histories["validation_losses"].append(mean_validation_loss)

        metric_histories["train_metrics"].append(train_metric)
        metric_histories["validation_metrics"].append(validation_metric)

        if scheduler is not None:
            scheduler.step()

        # Pasos de early stopping
        if mean_validation_loss <= best_valid_loss:
            # nos aseguramos de regresar el mejor modelo, no el último
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
            best_valid_loss = mean_validation_loss
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement == patience_epochs:
                print(f"Sin mejora en conjunto de validación por {patience_epochs} épocas")
                print(f"Parando entrenamiento en época {epoch}")

                break
        
    model.load_state_dict(torch.load(checkpoint_path))
    return loss_histories, metric_histories
    